# 03 Score: `gold.forecast` and `gold.recommendations`

- **`gold.forecast`**: the `@champion` model scores every feature row (batch, Spark UDF). Lost
  sales per store-SKU-day = forecast x the share of the day's demand profile that falls in
  trading hours ending with zero on hand. The hour-of-day profile is measured from sale lines on
  uncensored training days.
- **`gold.recommendations`**: tonight's order per store x SKU from the latest 15-minute on-hand
  snapshot, a 15-day forecast (today + 14) and the policy in `bigbox_ml/policy.py` /
  `policy.yaml` (the same code policy (b) runs in the backtest).

`mode=full` scores every day; `mode=incremental` (the 15-minute refresh) re-scores the last days.

In [ ]:
import datetime as dt
import json
import time

import mlflow
import numpy as np
import pandas as pd
import yaml
from mlflow.tracking import MlflowClient
from pyspark.sql import functions as F

from bigbox_ml import policy as pol
from bigbox_ml.features import FEATURES, KEYS, horizon_frame, planned_promotions

dbutils.widgets.text("catalog", "bigbox_fe_bar")
dbutils.widgets.dropdown("mode", "full", ["full", "incremental"])
CAT = dbutils.widgets.get("catalog")
MODE = dbutils.widgets.get("mode")
FT, MODEL = f"{CAT}.ml.store_sku_features", f"{CAT}.ml.demand_lgbm"
LATE_DAYS = 4

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient(registry_uri="databricks-uc")
try:
    VERSION = client.get_model_version_by_alias(MODEL, "champion").version
except Exception as e:  # first deploy: bigbox_ml_train has not registered a model yet
    dbutils.notebook.exit(json.dumps({"skipped": f"no @champion for {MODEL}: {type(e).__name__}"}))
MODEL_URI = f"models:/{MODEL}/{VERSION}"
P = pol.Policy.load("policy.yaml")
print(f"scoring with {MODEL} version {VERSION} (@champion), mode={MODE}")

scoring with bigbox_fe_bar.ml.demand_lgbm version 1 (@champion), mode=full


## Hour-of-day demand profile (measured)

In [ ]:
daily = spark.read.table(f"{CAT}.gold.store_sku_daily")
full_day = daily.agg(F.max("trading_hours")).first()[0]
with open("../generator/config.yaml") as fh:
    holdout_start = yaml.safe_load(fh)["calendar"]["history_end"] - dt.timedelta(days=55)
clean_days = daily.where(
    (F.col("stockout_hours") == 0)
    & (F.col("trading_hours") == full_day)
    & (F.col("date") < F.lit(holdout_start))
).select("store_id", "sku_id", "date")
pos = spark.read.table(f"{CAT}.silver.pos_order_lines").select(
    "store_id",
    "sku_id",
    F.col("sale_date").alias("date"),
    F.hour("event_ts").alias("hour"),
    F.col("quantity").alias("u"),
)
web = spark.read.table(f"{CAT}.silver.online_order_lines").select(
    "store_id",
    "sku_id",
    F.col("order_date").alias("date"),
    F.hour("order_ts").alias("hour"),
    F.col("requested_qty").alias("u"),
)
prof = (
    pos.unionByName(web)
    .join(clean_days, KEYS)
    .groupBy("hour")
    .agg(F.sum("u").alias("u"))
    .orderBy("hour")
    .toPandas()
)
prof["share"] = prof["u"] / prof["u"].sum()
PROFILE = dict(zip(prof["hour"].astype(int), prof["share"], strict=True))
print("demand share by trading hour (start hour):", {h: round(s, 4) for h, s in PROFILE.items()})
profile_sdf = spark.createDataFrame(
    [(h + 1, float(s)) for h, s in PROFILE.items()], "snap_hour INT, share DOUBLE"
)

demand share by trading hour (start hour): {8: 0.03, 9: 0.04, 10: 0.0601, 11: 0.0799, 12: 0.09, 13: 0.07, 14: 0.06, 15: 0.0601, 16: 0.07, 17: 0.0901, 18: 0.0999, 19: 0.1, 20: 0.0899, 21: 0.06}


## Batch scoring -> `gold.forecast`

In [ ]:
t0 = time.time()
feats = spark.read.table(FT)
start = None
if MODE == "incremental":
    start = feats.agg(F.max("date")).first()[0] - dt.timedelta(days=LATE_DAYS)
    feats = feats.where(F.col("date") >= F.lit(start))
predict = mlflow.pyfunc.spark_udf(spark, MODEL_URI, env_manager="local", result_type="double")
# The snapshot at hour h + 1 closes trading hour h (store-local clock).
snaps = spark.read.table(f"{CAT}.silver.inventory_snapshots")
if start is not None:
    snaps = snaps.where(F.col("snapshot_date") >= F.lit(start))
stocked_out = (
    snaps.where(F.col("on_hand") == 0)
    .join(profile_sdf, F.hour("snapshot_ts") == F.col("snap_hour"))
    .groupBy("store_id", "sku_id", F.col("snapshot_date").alias("date"))
    .agg(F.sum("share").alias("stockout_demand_share"))
)
actuals = daily.select(
    *KEYS,
    F.when(
        (F.col("stockout_hours") == 0) & (F.col("trading_hours") == full_day),
        F.col("units_sold").cast("double"),
    ).alias("uncensored_actual"),
)
scored = (
    feats.withColumn("forecast_units", predict(F.struct(*FEATURES)))
    .join(stocked_out, KEYS, "left")
    .join(actuals, KEYS, "left")
    .select(
        *KEYS,
        "region",
        "forecast_units",
        "uncensored_actual",
        F.coalesce("stockout_demand_share", F.lit(0.0)).alias("stockout_demand_share"),
        (F.col("forecast_units") * F.coalesce("stockout_demand_share", F.lit(0.0))).alias(
            "lost_sales_units"
        ),
        (
            F.col("forecast_units")
            * F.coalesce("stockout_demand_share", F.lit(0.0))
            * F.col("unit_price")
        ).alias("lost_sales"),
        F.lit(str(VERSION)).alias("model_version"),
        F.current_timestamp().alias("scored_at"),
    )
)
scored.createOrReplaceTempView("scored")
if start is None:
    spark.sql(f"INSERT OVERWRITE {CAT}.gold.forecast SELECT * FROM scored")
else:
    spark.sql(f"""MERGE INTO {CAT}.gold.forecast t USING scored s
        ON t.store_id = s.store_id AND t.sku_id = s.sku_id AND t.date = s.date
        WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")
fc = spark.read.table(f"{CAT}.gold.forecast")
print(f"gold.forecast written in {time.time() - t0:.0f} s")
display(
    fc.agg(
        F.count("*").alias("rows"),
        F.min("date").alias("first_date"),
        F.max("date").alias("last_date"),
        F.sum("lost_sales_units").alias("lost_sales_units"),
        F.sum("lost_sales").alias("lost_sales"),
    )
)

Unexpected status resolving file endpoints (403, request ID 1487b780-cb2e-4077-b32c-0422f4ebae30), using the default endpoints.


{'text/plain': 'Downloading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]', 'application/vnd.jupyter.widget-view+json': {'version_major': 2, 'version_minor': 0, 'model_id': '505312ca8bf7483686498c019cf94c99'}}

2026/11/24 08:10:10 WARNING mlflow.pyfunc: Calling `spark_udf()` with `env_manager="local"` does not recreate the same environment that was used during training, which may lead to errors or inaccurate predictions. We recommend specifying `env_manager="conda"`, which automatically recreates the environment that was used to train the model and performs inference in the recreated environment.


{'text/plain': 'Downloading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]', 'application/vnd.jupyter.widget-view+json': {'version_major': 2, 'version_minor': 0, 'model_id': '45052856cfc949f3b5227d2c4d8aef71'}}

2026/11/24 08:10:10 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


gold.forecast written in 121 s


| rows | first_date | last_date | lost_sales_units | lost_sales |
|---|---|---|---|---|
| 10108926 | 2025-10-01 | 2026-10-01 | 1093446.50143 | 19905127.66302139 |


## Tonight's recommendations -> `gold.recommendations`

In [ ]:
t0 = time.time()
today = spark.read.table(FT).agg(F.max("date")).first()[0]
origin = spark.read.table(FT).where(F.col("date") == F.lit(today)).toPandas()
K = P.horizon_days + 1  # today + horizon_days
promos = planned_promotions(
    spark.read.table(f"{CAT}.silver.promotions"),
    spark.read.table(f"{CAT}.silver.stores"),
    str(today),
    str(today + dt.timedelta(days=K)),
).toPandas()
h = horizon_frame(origin, promos, K)
model = mlflow.pyfunc.load_model(MODEL_URI)
f = np.asarray(model.predict(h[FEATURES]), dtype=float).reshape(len(origin), K)
pos_now = (
    spark.read.table(f"{CAT}.gold.inventory_position")
    .select("store_id", "sku_id", "on_hand", "on_order_units", "as_of_ts")
    .toPandas()
)
o = origin.merge(pos_now, on=["store_id", "sku_id"], how="left", validate="one_to_one")
o["on_hand"] = o["on_hand"].fillna(0).astype(np.int64)
o["on_order_units"] = o["on_order_units"].fillna(0).astype(np.int64)
as_of_hour = pd.to_datetime(o["as_of_ts"]).dt.hour.fillna(24).to_numpy()  # no snapshot: day over
remaining = np.array([sum(s for hr, s in PROFILE.items() if hr >= a) for a in as_of_hour])

lead = o["lead_time_days"].to_numpy().astype(np.int64)
case = o["case_pack"].to_numpy().astype(np.int64)
shelf = o["shelf_life_days"].to_numpy().astype(np.int64)
perish = o["perishable"].to_numpy() > 0
on_hand, on_order = o["on_hand"].to_numpy(), o["on_order_units"].to_numpy()
rest_today = f[:, 0] * remaining
future = f[:, 1:]  # tonight's order covers tomorrow onwards
sigma = pol.daily_sigma(future[:, 0], o["std_28"].to_numpy())
rop, out = pol.levels(future, lead, shelf, perish, sigma, case, P)
projected = on_hand + on_order - rest_today  # position at tonight's close
qty = pol.order_quantity(projected, rop, out, case)
mu = rest_today + np.cumsum(future, axis=1)[np.arange(len(o)), lead - 1]
window_sigma = sigma * np.sqrt(lead + remaining)
avail = on_hand + on_order
risk = pol.stockout_risk(avail, mu, window_sigma)
short = pol.expected_shortfall(avail, mu, window_sigma)
next7 = future[:, :7].sum(axis=1)
dos = pol.days_of_supply(on_hand, next7 / 7)
over = pol.overstock(on_hand, dos, perish, shelf, P)
costs = spark.read.table(f"{CAT}.silver.products").select("sku_id", "unit_cost").toPandas()
o = o.merge(costs, on="sku_id", how="left", validate="many_to_one")
price = o["unit_price"].to_numpy()
reason = np.where(risk >= P.at_risk_threshold, "STOCKOUT_RISK", np.where(over, "OVERSTOCK", None))
rec = pd.DataFrame(
    {
        "store_id": o["store_id"],
        "sku_id": o["sku_id"],
        "region": o["region"],
        "format": o["format"],
        "category": o["category"],
        "perishable": perish,
        "as_of_ts": o["as_of_ts"],
        "forecast_date": today,
        "on_hand": on_hand,
        "on_order_units": on_order,
        "lead_time_days": lead.astype(np.int32),
        "case_pack": case.astype(np.int32),
        "shelf_life_days": shelf.astype(np.int32),
        "unit_price": price,
        "unit_cost": o["unit_cost"],
        "forecast_today": f[:, 0],
        "forecast_next_7d": next7,
        "demand_until_delivery": mu,
        "sigma_daily": sigma,
        "stockout_risk": risk,
        "expected_lost_units": short,
        "expected_lost_sales": short * price,
        "days_of_supply": pd.Series(dos).replace({np.nan: None}),
        "overstock_flag": over,
        "reorder_point": rop,
        "order_up_to": out,
        "recommended_order_qty": qty,
        "recommended_order_value": qty * o["unit_cost"].to_numpy(),
        "queue_reason": reason,
    }
)
rec["priority"] = rec["expected_lost_sales"].rank(ascending=False, method="first").astype(np.int32)
rec["features_json"] = [
    json.dumps({k: (None if pd.isna(v) else v) for k, v in r.items()})
    for r in origin[FEATURES].to_dict(orient="records")
]
rec["model_version"] = str(VERSION)
rec["scored_at"] = dt.datetime.now(dt.UTC)
cols = [c.name for c in spark.read.table(f"{CAT}.gold.recommendations").schema]
spark.createDataFrame(rec[cols]).createOrReplaceTempView("rec")
spark.sql(f"INSERT OVERWRITE {CAT}.gold.recommendations SELECT * FROM rec")
print(f"gold.recommendations: {len(rec):,} rows for {today} in {time.time() - t0:.0f} s")
print(rec["queue_reason"].value_counts(dropna=False).to_string())

Unexpected status resolving file endpoints (403, request ID ce00d060-d1d2-4e25-99e6-b3c7b65666ef), using the default endpoints.


{'text/plain': 'Downloading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]', 'application/vnd.jupyter.widget-view+json': {'version_major': 2, 'version_minor': 0, 'model_id': '0a06c4d449144e61935a41d4023e44d3'}}

gold.recommendations: 28,005 rows for 2026-10-01 in 23 s
queue_reason
None             22872
OVERSTOCK         4003
STOCKOUT_RISK     1130


In [ ]:
display(
    spark.read.table(f"{CAT}.gold.recommendations")
    .where("queue_reason IS NOT NULL")
    .orderBy("priority")
    .select(
        "priority",
        "store_id",
        "sku_id",
        "category",
        "on_hand",
        "on_order_units",
        "demand_until_delivery",
        "stockout_risk",
        "expected_lost_sales",
        "recommended_order_qty",
        "queue_reason",
    )
    .limit(10)
)

| priority | store_id | sku_id | category | on_hand | on_order_units | demand_until_delivery | stockout_risk | expected_lost_sales | recommended_order_qty | queue_reason |
|---|---|---|---|---|---|---|---|---|---|---|
| 1 | S073 | SKU0263 | Toys | 12 | 42 | 141.6239739511364 | 0.9999814848328261 | 1401.1087263040351 | 144 | STOCKOUT_RISK |
| 2 | S048 | SKU0263 | Toys | 0 | 54 | 137.43267754855754 | 0.9999303294454754 | 1334.094244077608 | 138 | STOCKOUT_RISK |
| 3 | S099 | SKU0263 | Toys | 5 | 66 | 152.05571032354646 | 0.9988776658064142 | 1296.213719443597 | 144 | STOCKOUT_RISK |
| 4 | S093 | SKU0263 | Toys | 10 | 54 | 143.25969463731505 | 0.9999316945428487 | 1267.3678416580808 | 138 | STOCKOUT_RISK |
| 5 | S097 | SKU0263 | Toys | 8 | 54 | 136.38857732388823 | 0.9997395137658213 | 1189.4959550580943 | 132 | STOCKOUT_RISK |
| 6 | S016 | SKU0263 | Toys | 13 | 84 | 168.87548652871263 | 0.9966981744798189 | 1149.7143949185036 | 144 | STOCKOUT_RISK |
| 7 | S032 | SKU0263 | Toys | 30 | 48 | 146.5721481377942 | 0.9990679618335637 | 1096.5591311223543 | 126 | STOCKOUT_RISK |
| 8 | S036 | SKU0263 | Toys | 0 | 90 | 157.5747931755578 | 0.9975638328904282 | 1080.7983395623992 | 132 | STOCKOUT_RISK |
| 9 | S081 | SKU0263 | Toys | 0 | 120 | 185.929309222759 | 0.9871864420644733 | 1056.3173952187767 | 144 | STOCKOUT_RISK |
| 10 | S065 | SKU0253 | Toys | 2 | 18 | 40.53533144272626 | 0.9971516295655802 | 1047.4208408220986 | 42 | STOCKOUT_RISK |


In [ ]:
summary = fc.agg(F.count("*").alias("n"), F.max("date").alias("d")).first()
dbutils.notebook.exit(
    json.dumps(
        {
            "model_version": str(VERSION),
            "mode": MODE,
            "forecast_rows": summary["n"],
            "forecast_last_date": str(summary["d"]),
            "recommendations": len(rec),
            "forecast_date": str(today),
            "queue": {
                str(k): int(v) for k, v in rec["queue_reason"].value_counts(dropna=False).items()
            },
            "hour_profile": {str(k): round(v, 6) for k, v in PROFILE.items()},
        }
    )
)

{"model_version": "1", "mode": "full", "forecast_rows": 10108926, "forecast_last_date": "2026-10-01", "recommendations": 28005, "forecast_date": "2026-10-01", "queue": {"None": 22872, "OVERSTOCK": 4003, "STOCKOUT_RISK": 1130}, "hour_profile": {"8": 0.030016, "9": 0.039997, "10": 0.060069, "11": 0.07991, "12": 0.089981, "13": 0.07001, "14": 0.060017, "15": 0.060069, "16": 0.069954, "17": 0.090107, "18": 0.099937, "19": 0.099983, "20": 0.089913, "21": 0.060038}}